In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis Resume (`emcee`): LBCO, HRPT

This tutorial shows how to reopen the Bayesian project created previously,
inspect the saved fit results and then run more sampling steps to
extend the existing chain. Both emcee and BUMPS-DREAM support saving
and resuming their sampler state, so the same workflow applies to
either engine.

This workflow is useful when:
- the initial sampling run has not yet converged and more steps are needed,
- the initial sampling run has converged but more steps are desired
  for better posterior resolution,
- the initial sampling run has converged but the posterior plots have
  not yet been inspected and the user wants to see the plots before
  deciding whether to run more steps.

The workflow uses the same La0.5Ba0.5CoO3 powder diffraction example
as the DREAM Bayesian tutorial:

- run a short local refinement,
- derive finite fit bounds for the sampled parameters,
- switch to emcee and sample the posterior,
- save the project with the emcee chain,
- resume the chain with additional steps,
- inspect posterior plots after each sampling stage.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📂 Load Project

### Locate Project

Download and extract the saved emcee project, with the persisted chain
and posterior caches, from the EasyDiffraction data repository.

In [3]:
project_dir = edi.download_data('proj-lbco-hrpt-emcee', destination='projects')

Getting data...


Data 'proj-lbco-hrpt-emcee': Bayesian Analysis (emcee): LBCO, HRPT


✅ Data 'proj-lbco-hrpt-emcee' downloaded and extracted to '../../../projects/proj-lbco-hrpt-emcee-a84961b91bad'


### Load Project

Loading restores the persisted fit state, posterior samples, and plot
caches. No new fit is launched in this tutorial.

In [4]:
project = edi.Project.load(project_dir)

⚠️ Switching minimizer type removes these settings:                                                                               
   • max_iterations                                                                                                               


⚠️ Switching minimizer type adds these settings with defaults:                                                                    
   • burn_in_steps=1000                                                                                                           
   • initialization_method='ball'                                                                                                 
   • parallel_workers=0                                                                                                           
   • population_size=32                                                                                                           
   • proposal_moves='de'                                                                                                          
   • random_seed=None                                                                                                             
   • sampling_steps=5000                                                           

Re-save the project to a fresh working directory so resuming the
chain below writes there instead of the bundled read-only copy.

In [5]:
project.save_as(dir_path='projects/bayesian-emcee-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_emcee' to '../../../projects/bayesian-emcee-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html


## 📊 Inspect Results

### Display Structure

Render the La0.5Ba0.5CoO3 structure restored from the saved project.

In [6]:
project.display.structure(struct_name='lbco')

Structure 🧩 'lbco' (Atom view type: 'covalent')


### Display Fit Results

The fit summary reports the committed point estimate, sampler
settings, convergence diagnostics, and posterior parameter summaries
from the saved Bayesian run.

In [7]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,10000,Total sampler iterations per chain.
2,burn_in_steps,2000,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,16,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,ball,emcee walker initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.
8,proposal_moves,de,Single emcee proposal move; move mixtures are not persisted in v1.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,emcee
2,✅ Overall status,success
3,💬 Engine message,emcee sampling completed
4,⏱️ Fitting time (seconds),2224.84
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.66
7,"📏 R-factor squared (Rf², %)",4.92
8,"📏 Weighted R-factor (wR, %)",4.09
9,📉 Best log-posterior,-1157.00
10,📊 Convergence status,passed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↓
2,hrpt,linked_structure,lbco,scale,,9.1329,9.1340,0.0290,0.01 % ↑
3,hrpt,peak,,broad_gauss_u,deg²,0.0817,0.0813,0.0067,0.45 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1167,0.0048,0.18 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6306,0.6303,0.0017,0.04 % ↓


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.001,8943.4
2,hrpt,linked_structure,lbco,scale,,9.1329,"[9.0760, 9.1898]",1.002,9133.2
3,hrpt,peak,,broad_gauss_u,deg²,0.0816,"[0.0686, 0.0947]",1.002,8749.1
4,hrpt,peak,,broad_gauss_v,deg²,-0.1168,"[-0.1262, -0.1075]",1.002,8728.3
5,hrpt,instrument,,twotheta_offset,deg,0.6303,"[0.6269, 0.6337]",1.001,8838.7


### Display Correlations

The correlation matrix is restored from the saved project state.

In [8]:
project.display.fit.correlations()

### Display Posterior Densities

The pair plot and one-dimensional posterior distributions now load
from the persisted caches generated when the Bayesian fit was saved.

In [9]:
project.display.posterior.pairs()

In [10]:
project.display.posterior.distribution()

### Display Posterior Predictive

The posterior predictive view reuses the cached predictive summary
stored in the project rather than recalculating it on first display.
It overlays the 95% credible interval propagated from the posterior
samples.

In [11]:
project.display.posterior.predictive(expt_name='hrpt')

A zoomed view is useful for checking the propagated uncertainty in a
narrow region of the diffraction pattern.

In [12]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 🎲 Resume Sampling

### Run Sampling

Resume from the saved backend and append 100 more emcee steps to the
existing chain. We use only 100 steps here to keep the tutorial fast,
but in practice you would typically run more steps to ensure
convergence and better posterior resolution.

In [13]:
project.analysis.minimizer.random_seed = 42  # fixed seed for reproducible output
project.analysis.fit(resume=True, extra_steps=100)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'emcee'...


📈 Bayesian sampling progress:


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html


In [14]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,10000,Total sampler iterations per chain.
2,burn_in_steps,2000,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,16,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,ball,emcee walker initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.
8,proposal_moves,de,Single emcee proposal move; move mixtures are not persisted in v1.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,emcee
2,✅ Overall status,success
3,💬 Engine message,emcee sampling completed
4,⏱️ Fitting time (seconds),72.56
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.65
7,"📏 R-factor squared (Rf², %)",4.92
8,"📏 Weighted R-factor (wR, %)",4.09
9,📉 Best log-posterior,-1157.00
10,📊 Convergence status,passed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↓
2,hrpt,linked_structure,lbco,scale,,9.1340,9.1337,0.0290,0.00 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.0813,0.0813,0.0067,0.05 % ↑
4,hrpt,peak,,broad_gauss_v,deg²,-0.1167,-0.1167,0.0048,0.01 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6303,0.6303,0.0017,0.00 % ↓


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.001,9038.2
2,hrpt,linked_structure,lbco,scale,,9.1329,"[9.0761, 9.1899]",1.002,9260.6
3,hrpt,peak,,broad_gauss_u,deg²,0.0816,"[0.0686, 0.0947]",1.002,8830.2
4,hrpt,peak,,broad_gauss_v,deg²,-0.1168,"[-0.1262, -0.1075]",1.002,8806.6
5,hrpt,instrument,,twotheta_offset,deg,0.6303,"[0.6269, 0.6337]",1.001,8922.9


### Display Resumed Posterior

After resume, the posterior plots use the extended chain.

In [15]:
project.display.posterior.pairs()

In [16]:
project.display.posterior.distribution()

In [17]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

## 💾 Save Project

In [18]:
project.save_as(dir_path='projects/bayesian-emcee-resume-lbco-hrpt')

Saving project 📦 'lbco_hrpt_emcee' to '../../../projects/bayesian-emcee-resume-lbco-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 lbco_hrpt_emcee.html
